In [1]:
import os
import ssl

from google.adk.agents import Agent
from google.adk.models import Gemini
from google.adk.runners import InMemoryRunner
from google.genai import types

In [2]:
# disable ssl verification on ADK
# not tested on hf proxy yet

ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE



In [13]:
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "projectid",)
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "global",)
MODEL_ID = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite",)

# Tell ADK / Google GenAI to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print("Project :", PROJECT_ID)
print("Location:", LOCATION)
print("Model   :", MODEL_ID)

Project : yourprojectid
Location: global
Model   : gemini-2.5-flash-lite


In [4]:

def add(a: float, b: float) -> float:
    """Add two numbers."""

    print("\n>>> Python add() called")
    print(">>> a =", a)
    print(">>> b =", b)

    result = a + b

    print(">>> result =", result)

    return result


def sub(a: float, b: float) -> float:
    """Subtract b from a."""

    print("\n>>> Python sub() called")
    print(">>> a =", a)
    print(">>> b =", b)

    result = a - b

    print(">>> result =", result)

    return result

In [5]:
root_agent = Agent(
    name="calculator_agent",

    model=Gemini(
        model=MODEL_ID,
        http_options=types.HttpOptions(
            client_args={
                "verify": ssl_context,
            },
            async_client_args={
                "verify": ssl_context,
            },
        ),
    ),

    instruction="""
You are a simple calculator agent.

Rules:
- For addition, use the add tool.
- For subtraction, use the sub tool.
- Always use a tool instead of calculating the answer yourself.
- Return the final result clearly.
""",

    tools=[
        add,
        sub,
    ],
)

In [7]:

runner = InMemoryRunner(
    agent=root_agent,
    app_name="calculator_app",
)


In [10]:
question = "What is 25 plus 17?"

print("Question:", question)

response = await runner.run_debug(
    question
)

print ("--")


Question: What is 25 plus 17?

>>> Python add() called
>>> a = 25
>>> b = 17
>>> result = 42
calculator_agent > The sum of 25 and 17 is 42.
--


In [ ]:

print (response)